In [48]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [49]:
# Importações
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class LoadData:
  def __init__(self, base_path, batch_size = 32 ):
    self.base_path = base_path
    self.batch_size = batch_size
    self.transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
    ])
    self.train_loader = self.create_loader('train', shuffle = True)
    self.test_loader = self.create_loader('test', shuffle = False)
    self.val_loader = self.create_loader('val', shuffle = False)

  def create_loader(self,split_name, shuffle):
    folder_path = os.path.join(self.base_path, split_name)
    dataset = datasets.ImageFolder(root=folder_path, transform=self.transform)
    return DataLoader(dataset, batch_size=self.batch_size, shuffle=shuffle)


In [50]:
import torch
import torch.nn as nn
import torch.optim as optim

class CNNmodel(nn.Module):
    def __init__(self):
        super(CNNmodel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Linear(32 * 56 * 56, 2)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def treinar(self, data_manager, epochs=5):
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)

        print(f"Iniciando treinamento no dispositivo: {device}")

        for epoch in range(epochs):
            self.train()
            total_loss = 0

            for images, labels in data_manager.train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = self(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"Época [{epoch+1}/{epochs}] - Perda (Loss): {total_loss/len(data_manager.train_loader):.4f}")

    def avaliar(self, data_manager):
        self.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)
        total_acertos = 0
        total_imagens = 0

        print("\nIniciando avaliação no dataset de teste...")

        with torch.no_grad():
            for images, labels in data_manager.test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = self(images)
                _, palpites = torch.max(outputs, 1)
                total_imagens += labels.size(0)
                total_acertos += (palpites == labels).sum().item()

        print(f"Acurácia Final no Teste: {100 * total_acertos / total_imagens:.2f}%")

In [51]:

dados = LoadData(base_path="/kaggle/input/chest-xray-pneumonia/chest_xray", batch_size=32)
modelo = CNNmodel()
modelo.treinar(data_manager=dados, epochs=3)
modelo.avaliar(data_manager=dados)

Iniciando treinamento no dispositivo: cpu
Época [1/3] - Perda (Loss): 0.2789
Época [2/3] - Perda (Loss): 0.0694
Época [3/3] - Perda (Loss): 0.0454

Iniciando avaliação no dataset de teste...
Acurácia Final no Teste: 68.91%
